In [ ]:
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import yaml
import scanpy as sc

from torch.utils.data import DataLoader
from torch.optim import Adam

cwd1 = os.path.dirname(os.getcwd())
sys.path.append(cwd1)
cwd = Path.cwd()

from scmivae import iVAE, CountDataset, ModelTrainerIVAE
from scmivae.inferencer import VAEInference

In [5]:
UMAP_PARAMS = {
    'v1':  {'n_components': 2,  'n_neighbors': 100, 'min_dist': 0.9, 'random_state': 41},
}
# hyper parameters
load_model = True

# 1. Data Sets

The `tutorial_data.h5ad` dataset has been subsampled from the retina data in the below citation:
* Karthik Shekhar, Sylvain W. Lapan, Irene E. Whitney, Nicholas M. Tran, Evan Z. Macosko, Monika Kowalczyk, Xian Adiconis, Joshua Z. Levin, James Nemesh, Melissa Goldman, Steven A. McCarroll, Constance L. Cepko, Aviv Regev, and Joshua R. Sanes. Comprehensive Classification of Retinal Bipolar Neurons by Single-Cell Transcriptomics. Cell 2016
* Accession: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE81904

In [6]:
adata = sc.read_h5ad('tutorial_data.h5ad')

In [7]:
# count version
scData = CountDataset(
    gene_expression = np.array(adata.layers['logcounts'].todense()),
    cat_df = adata.obs[['sample_collapsed']],
    cont_df = None,
    data_dist='gaussian'
)

train_loader = DataLoader(dataset=scData, shuffle=True, drop_last=True, batch_size=256, num_workers=0)

In [8]:
# to check reference batch encoding
pd.concat([
    adata.obs[['sample_collapsed']].reset_index(drop=True),
    pd.DataFrame(scData.cat_df_tensor).rename(columns={0: 'label'}).reset_index(drop=True)
], axis=1).drop_duplicates()


,sample_collapsed,label
0,Bipolar5-6,1
3,Bipolar1-4,0


# 2. VAE

In [9]:
model_folder = 'test_run'

model_dir = (cwd/'output'/model_folder)
model_dir.mkdir(parents=True, exist_ok=True)

In [10]:
def resolve_data_config(data_cfg, scData):
    """
    Resolves fields that depend on the loaded dataset.

    Replaces:
        data_dim: auto  -> scData.data_dim
        num_cats: auto  -> scData.num_cats
        cont_dim: auto  -> scData.cont_dim
    """

    resolved = data_cfg.copy()

    auto_map = {
        "data_dim": scData.data_dim,
        "num_cats": scData.num_cats,
        "cont_dim": scData.cont_dim,
    }

    for key, val in auto_map.items():
        if resolved.get(key) == "auto":
            resolved[key] = val

    return resolved

In [11]:
if not load_model:
    with open(cwd / 'base_model' / "train.yaml") as f:
        tmp_cfg = yaml.safe_load(f)
    device_config = tmp_cfg["device"]
    train_config  = tmp_cfg["train"]

    device_config['device'] = 'mps'
    train_config['rec_num_rows'] = int(np.ceil(np.sqrt(scData.data_dim)))

    # set data config
    with open(cwd / 'base_model' / "data.yaml") as f:
        data_config = yaml.safe_load(f)['data']

    data_config = resolve_data_config(data_config, scData)

    # set VAE, MLP, branching config
    with open(cwd / 'base_model' / "model.yaml") as f:
        tmp_cfg = yaml.safe_load(f)
    mlp_config = tmp_cfg["mlp"]
    vae_config  = tmp_cfg["vae"]

In [12]:
if not load_model:
    # make any overrides in the config
    vae_config['decoder_var'] = 0.00625
    train_config['epochs'] = 26

In [13]:
if not load_model:
    experiment_config = {
        "device_config": device_config,
        "data_config": data_config,
        "vae_config": vae_config,
        "mlp_config": mlp_config,
        "train_config": train_config,
        "model_dir": model_folder
    }

    with open(model_dir / "config.yaml", "w") as f:
        yaml.safe_dump(experiment_config, f, sort_keys=False)
else:
    # to load model and override settings
    with open(model_dir / "config.yaml") as f:
        cfg = yaml.safe_load(f)

    device_config = cfg["device_config"]
    data_config = cfg["data_config"]
    vae_config = cfg["vae_config"]
    mlp_config = cfg["mlp_config"]
    train_config = cfg["train_config"]

    device_config['device'] = 'cpu' # or which is available here

In [14]:
iVAE_model = iVAE(**device_config, **data_config, **vae_config, **mlp_config)
optimizer = Adam(iVAE_model.parameters(), lr=0.001)

# iVAE_model = iVAE(
#     **cfg["data_config"],
#     **cfg["vae_config"],
#     **cfg["mlp_config"],
#     **cfg["device_config"],
# )

Not using additive categorical embedding
{
  "dimensions": {
    "data_dim": 2000,
    "latent_dim_total": 16,
    "latent_dim_non_branching": 16,
    "hidden_dim": 64
  },
  "architecture": {
    "n_layers": 2,
    "activation": "lrelu",
    "dropout": 0.1,
    "batchnorm": true
  },
  "decoder": {
    "distribution": "gaussian",
    "variance_mode": "fixed",
    "fixed_variance": 0.0062500000931322575,
    "softmax_scaling": false,
    "log1p_transform": false
  },
  "prior": {
    "net_mode": "mean",
    "distribution": "Normal"
  },
  "conditioning": {
    "cvae": false,
    "cvae_dim": 0,
    "cvae_aux": false
  },
  "auxiliary_variables": {
    "categorical_dims": [
      2
    ],
    "continuous_dim": 0
  }
}


In [ ]:
torch.compile(iVAE_model)
# This will either load or reload the model
trainer = ModelTrainerIVAE(
    model = iVAE_model, optimizer=optimizer, load_model=load_model, resume_from_latest=True,
    early_stopping_start=train_config['early_stopping_start'], epoch_checkpoint_freq=train_config['epoch_checkpoint_freq'],
    checkpoint_path_latest=model_dir/'latest.pt', checkpoint_path_best=model_dir/'best.pt',
    model_dir = model_dir,
    device=device_config['device'], verbose=3,
)

if not load_model:
    trainer.train_ivae(
        device=device_config['device'],
        train_loader=train_loader, valid_loader=train_loader,
        rec_loader=train_loader, rec_samples=16, rec_cmap='viridis',
        **train_config,
    )

    trainer.plt_training_loss(epoch_threshold=4, plot_mode = 'analysis', test_loss=None, model_name=model_folder)


Loading existing (best) model ...



# 3. Inference

In [ ]:
samples = adata.obs.copy()
batch_column = 'sample_collapsed'
reference_batch = 'Bipolar1-4'

In [ ]:
# Initialize the inference handler
inferencer = VAEInference(
    trainer=trainer,
    dataset=scData,
    batch_size=256,
    num_workers=0,
    device='cpu',
    run_inferencer=True
)

# at initalisation the inferencer will check if categorical and continuous variables are available and make all of them equal
# ('corrected' for categorical variables and 'corrected_catcont' for continuous, setting all values to 1)
inferencer.datasets

{'default': <scmivae.utils_data.CountDataset at 0x1638ddc50>,
 'corrected': <scmivae.utils_data.CountDataset at 0x31c049390>}

In [32]:
# to perform manual correction, you can specify which columns of the scData.cat_df and scData.cont_df you want to modify directly
tmp_dataset = inferencer.add_manual_correction(cat_target=[0], cat_vals=[1], mode='override')
pd.DataFrame(tmp_dataset.cat_df_tensor).drop_duplicates()

,0
0,1


In [ ]:
# To perform counterfactual inference (eg batch correction or condition transfer)
# you need to specify the source and the target. Default corresponds to the original dataset
dataset_key = 'corrected'
inferencer.run_counterfactual_inference(dataset_key='corrected_override_[0]-[1]', dataset_ref='default')

In [ ]:
# counterfactual latent states (appended by ViaPrior)
inferencer.embeddings.keys()

dict_keys(['default', 'corrected', 'corrected_override_[0]-[1]', 'corrected_ViaPrior', 'corrected_override_[0]-[1]_ViaPrior'])

In [ ]:
# counterfactual expression states (appended by ViaPrior)
inferencer.decodings.keys()

dict_keys(['default', 'corrected', 'corrected_override_[0]-[1]', 'corrected_ViaPrior', 'corrected_override_[0]-[1]_ViaPrior'])

In [ ]:
# plot latent space and umap embeddings with coloured stratification
plot_configs = [('sample_collapsed', '_sample_collapsed'), ('CLUSTER', '_CLUSTER')]
correction_key = 'corrected_override_[0]-[1]_ViaPrior'

inferencer.plot_all(
    samples=samples,
    plot_configs=plot_configs,
    output_dir=model_dir / 'embeddings' / correction_key,
    dataset_key=correction_key,
    use_pca=False,
    pca_components='full',
    umap_params=UMAP_PARAMS['v1'],
    figsize_per_plot=8,
    plot_leiden=False
)

Computing UMAP for 'corrected_override_[0]-[1]_ViaPrior' (use_pca=False)...


/opt/homebrew/Caskroom/miniforge/base/envs/scmivae-test/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Generating plots for 'sample_collapsed' with suffix '_sample_collapsed'...
Generating plots for 'CLUSTER' with suffix '_CLUSTER'...
